# Exemplos Comparativos — Aula 11
## Técnicas Complementares: Multimodal, LLMLingua, ColBERT, Time-Aware, DSPy

**Aula 11 · MBA RAG & CAG Aplicados a Direito e Segurança Pública**

Este notebook contém exemplos didáticos isolados para cada técnica, sem necessidade de executar os labs completos. Use-os como referência rápida durante o projeto final.

---

## Exemplo 1 — Decay Exponencial (Time-Aware)

Cálculo manual da função de decay para documentos jurídicos de diferentes épocas.

In [ ]:
import math
from datetime import datetime

def decay_exp(data_doc: str, data_ref: str = "2024-12-01", scale: int = 365, offset: int = 30) -> float:
    """Função de decay exponencial. Equivale ao 'exp' do OpenSearch function_score."""
    doc_dt = datetime.strptime(data_doc, "%Y-%m-%d")
    ref_dt = datetime.strptime(data_ref, "%Y-%m-%d")
    age = (ref_dt - doc_dt).days
    lam = math.log(2) / scale
    return math.exp(-lam * max(0, age - offset))

# Exemplo: comparando texto do CP original com versão do Pacote Anticrime
casos = [
    ("CP original (1940)", "1940-12-07"),
    ("Reforma CP (1984)",  "1984-07-13"),
    ("Pacote Anticrime",   "2019-12-24"),
    ("Resolução CNJ 2024", "2024-01-17"),
]

print(f"{'Documento':<25} {'Data':>12} {'Decay (scale=365)':>18} {'Decay (scale=730)':>18}")
print("-" * 75)
for nome, data in casos:
    d365 = decay_exp(data, scale=365)
    d730 = decay_exp(data, scale=730)
    print(f"{nome:<25} {data:>12} {d365:>18.6f} {d730:>18.6f}")

## Exemplo 2 — MaxSim (ColBERT)

Demonstração da operação MaxSim que diferencia ColBERT de bi-encoders convencionais.

In [ ]:
import numpy as np

def maxsim_score(query_tokens: list[np.ndarray], doc_tokens: list[np.ndarray]) -> float:
    """
    Implementação do MaxSim: para cada token da query, encontra
    o token do documento com maior similaridade, soma os resultados.
    
    Fórmula:
      score = Σ_{qi in Q} max_{dj in D} cos(qi, dj)
    """
    score = 0.0
    for q_tok in query_tokens:
        # Similaridade coseno do token da query com todos os tokens do documento
        q_norm = q_tok / (np.linalg.norm(q_tok) + 1e-9)
        sims = [
            np.dot(q_norm, d_tok / (np.linalg.norm(d_tok) + 1e-9))
            for d_tok in doc_tokens
        ]
        score += max(sims)  # MaxSim: pega o maior
    return score


# Exemplo com vetores 2D para visualização
np.random.seed(42)

# Simulação: tokens de uma query sobre "arma de fogo"
q_arma = np.array([0.9, 0.1])    # "arma"
q_fogo = np.array([0.1, 0.9])    # "fogo"

# Documento 1: "suspeito portava arma" — relevante
d1 = [
    np.array([0.1, 0.2]),  # "suspeito"
    np.array([0.2, 0.1]),  # "portava"
    np.array([0.85, 0.15]),  # "arma"
]

# Documento 2: "disparo de revólver" — relacionado mas diferente
d2 = [
    np.array([0.4, 0.4]),  # "disparo"
    np.array([0.3, 0.3]),  # "de"
    np.array([0.5, 0.5]),  # "revólver"
]

query_tokens = [q_arma, q_fogo]

score_d1 = maxsim_score(query_tokens, d1)
score_d2 = maxsim_score(query_tokens, d2)

print("Demonstração MaxSim (ColBERT):")
print(f"  Query: ['arma', 'fogo']")
print(f"  Doc1 ('suspeito portava arma'): MaxSim = {score_d1:.4f}")
print(f"  Doc2 ('disparo de revólver'):   MaxSim = {score_d2:.4f}")
print(f"  Vencedor: Doc {'1' if score_d1 > score_d2 else '2'} ({'✅ Correto' if score_d1 > score_d2 else '⚠️ Incorreto'})")
print()
print("💡 O MaxSim encontra o token mais alinhado do doc para cada token da query,")
print("   permitindo matching granular que bi-encoders não conseguem.")

## Exemplo 3 — Compressão de Prompt (LLMLingua)

Demonstração conceitual do processo de token pruning baseado em perplexidade.

In [ ]:
# Exemplo conceitual de compressão por perplexidade
# Mostra quais tokens seriam mantidos vs removidos

TEXTO_ORIGINAL = (
    "O artigo 33 da Lei 11.343 de 2006 tipifica o crime de tráfico de "
    "entorpecentes e prevê pena de reclusão de 5 a 15 anos e multa para "
    "aquele que importar exportar remeter preparar produzir fabricar "
    "adquirir vender expor à venda oferecer drogas sem autorização."
)

# Perplexidade simulada (em prática: calculada por modelo pequeno como GPT-2)
# Alta perplexidade = token surpreendente = mantido
# Baixa perplexidade = token previsível = candidato à remoção
PERPLEXIDADES = {
    "O": 2.1, "artigo": 8.4, "33": 42.1, "da": 1.8, "Lei": 9.2,
    "11.343": 89.4, "de": 1.3, "2006": 67.8, "tipifica": 22.1,
    "o": 1.5, "crime": 12.3, "de": 1.3, "tráfico": 38.7, "de": 1.3,
    "entorpecentes": 45.2, "e": 1.7, "prevê": 14.5, "pena": 28.3,
    "de": 1.3, "reclusão": 52.1, "de": 1.3, "5": 71.4, "a": 1.6,
    "15": 64.8, "anos": 31.2, "e": 1.7, "multa": 38.9, "para": 3.2,
    "aquele": 4.1, "que": 2.8, "importar": 41.3, "exportar": 45.6,
    "remeter": 48.2, "preparar": 43.7, "produzir": 44.1, "fabricar": 46.8,
    "adquirir": 47.2, "vender": 38.9, "expor": 41.5, "à": 2.4,
    "venda": 28.1, "oferecer": 39.4, "drogas": 52.3, "sem": 8.9,
    "autorização": 41.7
}

LIMIAR_PERPLEXIDADE = 15.0  # Tokens abaixo desse valor são candidatos à remoção

tokens = TEXTO_ORIGINAL.split()
mantidos = []
removidos = []

for token in tokens:
    token_limpo = token.strip('.,;:\'"')
    perp = PERPLEXIDADES.get(token_limpo, 20.0)
    if perp >= LIMIAR_PERPLEXIDADE:
        mantidos.append(token)
    else:
        removidos.append(token)

texto_comprimido = " ".join(mantidos)

print("=" * 60)
print("EXEMPLO LLMLingua — Token Pruning por Perplexidade")
print("=" * 60)
print(f"\n📝 ORIGINAL ({len(tokens)} tokens):")
print(TEXTO_ORIGINAL)

print(f"\n✂️  COMPRIMIDO ({len(mantidos)} tokens):")
print(texto_comprimido)

print(f"\n❌ REMOVIDOS ({len(removidos)} tokens):")
print(" | ".join(removidos))

taxa = 1 - len(mantidos)/len(tokens)
print(f"\n📊 Taxa de compressão: {taxa:.1%}")
print("💡 Tokens removidos são previsíveis (baixa perplexidade): artigos,")
print("   preposições, conectivos — informação semântica é preservada.")

## Exemplo 4 — Assinatura DSPy

Demonstração da estrutura de uma Signature e como o DSPy a transforma em prompt.

In [ ]:
# Exemplos de Assinaturas DSPy para domínio jurídico
# Cada assinatura define entradas/saídas sem especificar o prompt

EXEMPLOS_ASSINATURAS = [
    {
        "nome": "ClassificacaoCrime",
        "entradas": {
            "fatos": "Descrição objetiva dos fatos do caso",
            "contexto_legal": "Artigos relevantes do Código Penal"
        },
        "saidas": {
            "tipo_penal": "Nome e artigo do crime configurado",
            "qualificadoras": "Causas de aumento ou qualificadoras aplicáveis",
            "pena_base": "Pena prevista no tipo penal identificado"
        },
        "uso": "Triagem automática de boletins de ocorrência"
    },
    {
        "nome": "AnaliseHabeasCorpus",
        "entradas": {
            "fundamentos_prisao": "Motivos declarados na ordem de prisão",
            "jurisprudencia": "Precedentes do STF/STJ relevantes",
            "situacao_processual": "Fase processual e condição do preso"
        },
        "saidas": {
            "ilegalidade_identificada": "Fundamento de ilegalidade se houver",
            "recomendacao": "Impetrar HC, relaxar prisão ou manter",
            "fundamentacao": "Base legal e jurisprudencial da recomendação"
        },
        "uso": "Suporte à decisão em análise de pedidos de HC"
    },
    {
        "nome": "RelatorioCriminalIntel",
        "entradas": {
            "relatorios_ocorrencia": "Conjunto de ROs do período",
            "dados_mapa": "Geolocalização das ocorrências",
            "historico_area": "Histórico criminal da área"
        },
        "saidas": {
            "padrao_crime": "Padrão identificado (tempo, local, MO)",
            "areas_risco": "Áreas com maior concentração de incidentes",
            "recomendacao_policiamento": "Estratégia de policiamento sugerida"
        },
        "uso": "Análise de inteligência para alocação de efetivo policial"
    }
]

print("=" * 60)
print("EXEMPLOS DE ASSINATURAS DSPy — DOMÍNIO JURÍDICO")
print("=" * 60)

for sig in EXEMPLOS_ASSINATURAS:
    print(f"\n📋 {sig['nome']}")
    print(f"   Uso: {sig['uso']}")
    print(f"   Entradas ({len(sig['entradas'])}):")
    for campo, desc in sig['entradas'].items():
        print(f"     • {campo}: {desc}")
    print(f"   Saídas ({len(sig['saidas'])}):")
    for campo, desc in sig['saidas'].items():
        print(f"     • {campo}: {desc}")

print("""
💡 Vantagem DSPy:
   O desenvolvedor define O QUÊ (entradas e saídas),
   o DSPy descobre COMO (prompt otimizado + exemplos few-shot).
""")

---

## Resumo dos Exemplos

| Técnica | Exemplo | Conceito-chave |
|---|---|---|
| Time-Aware | Decay exponencial | `exp(-λ × max(0, age - offset))` |
| ColBERT | MaxSim | `Σ max_j cos(qi, dj)` |
| LLMLingua | Token pruning | Perplexidade < limiar → remover |
| DSPy | Assinatura | Define entradas/saídas, não prompt |

---
*Aula 11 · MBA RAG & CAG Aplicados a Direito e Segurança Pública*